# Participant and fold difficulty
This is a **post-hoc diagnostic analysis** of the completed 36-month MCI prognosis experiments. It does **not retrain** any model.

Its purpose is to test the hypothesis raised by the capacity ablation: the differences between test folds may be driven more by **participant composition / case difficulty** than by network size.

The analysis proceeds at three levels:

1. **Fold difficulty** — seed-ensemble ROC AUC with stratified bootstrap confidence intervals, target prevalence, missing-modality composition, uncertainty, and prediction variability.
2. **Participant difficulty** — participants consistently correct, consistently wrong, or seed-sensitive across the five neural seeds; association with modality availability, feature missingness, and model uncertainty.
3. **Model disagreement** — overlap of difficult participants across the small, medium, and full architectures, plus an optional RBF-SVM comparison if its saved prediction files are available.

### Interpretation constraint

This is exploratory analysis performed after the outer-test results have already been inspected. It should be used to **understand failure modes and motivate future/external validation**, not to remove difficult participants, redesign folds, or select a new model using the same test data.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# JHub paths

PROJECT_ROOT = Path("/home/jovyan/thesis_data/adni_mri/adni_mri")
RUNS_ROOT = Path("/home/jovyan/thesis_runs")

INPUT_ROOT = (
    PROJECT_ROOT
    / "models"
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

ARCHITECTURE_ROOTS = {
    "small_1.13M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_small_v1_md050"
    ),
    "medium_2.51M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_medium_v1_md050"
    ),
    "full_4.43M": (
        RUNS_ROOT
        / "gated_cmt_fixed_equal_fusion_full_arch_md050"
    ),
}

PRIMARY_ARCHITECTURE = "medium_2.51M"

SEEDS = [17, 42, 73, 101, 137]
FOLDS = [0, 1, 2, 3, 4]
THRESHOLD = 0.50

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20261004

ANALYSIS_ROOT = RUNS_ROOT / "participant_difficulty_analysis_v1"
TABLES_DIR = ANALYSIS_ROOT / "tables"
FIGURES_DIR = ANALYSIS_ROOT / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Input root:", INPUT_ROOT)
print("Analysis output:", ANALYSIS_ROOT)
print("Primary architecture:", PRIMARY_ARCHITECTURE)
print("Seeds:", SEEDS)
print("Folds:", FOLDS)


## 1. Helper functions

The loaders below deliberately validate participant identity and targets before merging anything. They also use the actual prediction columns written by the 3MT–TMC worker:

- `RID`
- `TARGET`
- `FINAL_P_pMCI`
- `FINAL_UNCERTAINTY`
- `THREE_MT_P_pMCI`
- `THREE_MT_UNCERTAINTY`
- `TMC_P_pMCI`
- `TMC_UNCERTAINTY`
- modality counts

Secondary columns are optional; the core analysis only requires the primary prediction fields.


In [ ]:
def require_columns(frame, columns, context="dataframe"):
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise KeyError(
            f"{context}: missing required columns {missing}. "
            f"Available columns: {frame.columns.tolist()}"
        )

def safe_auc(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_prob))

def safe_ap(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    if np.unique(y_true).size < 2:
        return np.nan
    return float(average_precision_score(y_true, y_prob))

def stratified_bootstrap_auc_ci(
    y_true,
    y_prob,
    n_bootstrap=2000,
    random_seed=20261004,
):
    """Stratified participant bootstrap, preserving class counts."""
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    idx0 = np.flatnonzero(y_true == 0)
    idx1 = np.flatnonzero(y_true == 1)

    if len(idx0) == 0 or len(idx1) == 0:
        return {
            "auc": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
            "bootstrap_sd": np.nan,
        }

    rng = np.random.default_rng(random_seed)
    aucs = np.empty(n_bootstrap, dtype=float)

    for b in range(n_bootstrap):
        sample0 = rng.choice(idx0, size=len(idx0), replace=True)
        sample1 = rng.choice(idx1, size=len(idx1), replace=True)
        sample = np.concatenate([sample0, sample1])

        aucs[b] = roc_auc_score(
            y_true[sample],
            y_prob[sample],
        )

    return {
        "auc": float(roc_auc_score(y_true, y_prob)),
        "ci_low": float(np.quantile(aucs, 0.025)),
        "ci_high": float(np.quantile(aucs, 0.975)),
        "bootstrap_sd": float(np.std(aucs, ddof=1)),
    }

def load_neural_predictions(
    experiment_root,
    architecture_name,
    seeds=SEEDS,
    folds=FOLDS,
):
    """Load all available neural test predictions for one architecture."""
    frames = []

    for seed in seeds:
        for fold in folds:
            path = (
                experiment_root
                / f"seed_{seed}"
                / "mci_prognosis"
                / f"fold_{fold}"
                / "predictions"
                / "test_predictions.csv"
            )

            if not path.exists():
                continue

            frame = pd.read_csv(path)

            require_columns(
                frame,
                ["RID", "TARGET", "FINAL_P_pMCI"],
                context=str(path),
            )

            if frame["RID"].duplicated().any():
                raise ValueError(
                    f"Duplicate RID within one prediction file: {path}"
                )

            frame = frame.copy()
            frame["SEED"] = int(seed)
            frame["FOLD"] = int(fold)
            frame["ARCHITECTURE"] = architecture_name

            frames.append(frame)

    if not frames:
        raise FileNotFoundError(
            f"No test prediction files found under {experiment_root}"
        )

    predictions = pd.concat(frames, ignore_index=True)

    # One participant should appear once per seed across the five OOF test folds.
    duplicate_keys = predictions.duplicated(
        ["SEED", "RID"],
        keep=False,
    )
    if duplicate_keys.any():
        bad = predictions.loc[
            duplicate_keys,
            ["SEED", "FOLD", "RID"],
        ].sort_values(["SEED", "RID"])
        raise ValueError(
            "A participant appears more than once within a seed's OOF "
            f"predictions. Example:\n{bad.head(20)}"
        )

    return predictions

def aggregate_neural_by_participant(predictions):
    """Collapse five stochastic seeds into one participant-level diagnostic row."""
    require_columns(
        predictions,
        ["RID", "TARGET", "FOLD", "SEED", "FINAL_P_pMCI"],
        "neural predictions",
    )

    data = predictions.copy()

    data["FINAL_PRED_CLASS"] = (
        data["FINAL_P_pMCI"] >= THRESHOLD
    ).astype(int)

    data["FINAL_CORRECT"] = (
        data["FINAL_PRED_CLASS"] == data["TARGET"]
    ).astype(int)

    if "THREE_MT_P_pMCI" in data.columns:
        data["THREE_MT_PRED_CLASS"] = (
            data["THREE_MT_P_pMCI"] >= THRESHOLD
        ).astype(int)

    if "TMC_P_pMCI" in data.columns:
        data["TMC_PRED_CLASS"] = (
            data["TMC_P_pMCI"] >= THRESHOLD
        ).astype(int)

    if (
        "THREE_MT_PRED_CLASS" in data.columns
        and "TMC_PRED_CLASS" in data.columns
    ):
        data["THREE_MT_TMC_CLASS_DISAGREE"] = (
            data["THREE_MT_PRED_CLASS"]
            != data["TMC_PRED_CLASS"]
        ).astype(int)

    agg_spec = {
        "TARGET": "first",
        "FOLD": "first",
        "SEED": "nunique",
        "FINAL_P_pMCI": ["mean", "std", "min", "max"],
        "FINAL_CORRECT": ["mean", "sum"],
    }

    optional_mean_std = [
        "FINAL_UNCERTAINTY",
        "THREE_MT_P_pMCI",
        "THREE_MT_UNCERTAINTY",
        "TMC_P_pMCI",
        "TMC_UNCERTAINTY",
        "ORIGINAL_MODALITY_COUNT",
        "EFFECTIVE_MODALITY_COUNT",
    ]

    for col in optional_mean_std:
        if col in data.columns:
            agg_spec[col] = ["mean", "std"]

    if "THREE_MT_TMC_CLASS_DISAGREE" in data.columns:
        agg_spec["THREE_MT_TMC_CLASS_DISAGREE"] = "mean"

    participant = data.groupby("RID").agg(agg_spec)
    participant.columns = [
        "__".join([str(x) for x in tup if str(x)])
        if isinstance(tup, tuple)
        else str(tup)
        for tup in participant.columns
    ]
    participant = participant.reset_index()

    rename_map = {
        "TARGET__first": "TARGET",
        "FOLD__first": "FOLD",
        "SEED__nunique": "N_SEEDS",
        "FINAL_P_pMCI__mean": "FINAL_P_MEAN",
        "FINAL_P_pMCI__std": "FINAL_P_SD",
        "FINAL_P_pMCI__min": "FINAL_P_MIN",
        "FINAL_P_pMCI__max": "FINAL_P_MAX",
        "FINAL_CORRECT__mean": "SEED_CORRECT_RATE",
        "FINAL_CORRECT__sum": "N_SEEDS_CORRECT",
        "FINAL_UNCERTAINTY__mean": "FINAL_UNCERTAINTY_MEAN",
        "FINAL_UNCERTAINTY__std": "FINAL_UNCERTAINTY_SD",
        "THREE_MT_P_pMCI__mean": "THREE_MT_P_MEAN",
        "THREE_MT_P_pMCI__std": "THREE_MT_P_SD",
        "THREE_MT_UNCERTAINTY__mean": "THREE_MT_UNCERTAINTY_MEAN",
        "THREE_MT_UNCERTAINTY__std": "THREE_MT_UNCERTAINTY_SD",
        "TMC_P_pMCI__mean": "TMC_P_MEAN",
        "TMC_P_pMCI__std": "TMC_P_SD",
        "TMC_UNCERTAINTY__mean": "TMC_UNCERTAINTY_MEAN",
        "TMC_UNCERTAINTY__std": "TMC_UNCERTAINTY_SD",
        "ORIGINAL_MODALITY_COUNT__mean": "ORIGINAL_MODALITY_COUNT_MEAN",
        "ORIGINAL_MODALITY_COUNT__std": "ORIGINAL_MODALITY_COUNT_SD",
        "EFFECTIVE_MODALITY_COUNT__mean": "EFFECTIVE_MODALITY_COUNT_MEAN",
        "EFFECTIVE_MODALITY_COUNT__std": "EFFECTIVE_MODALITY_COUNT_SD",
        "THREE_MT_TMC_CLASS_DISAGREE__mean":
            "THREE_MT_TMC_CLASS_DISAGREE_RATE",
    }
    participant = participant.rename(columns=rename_map)

    if not (participant["N_SEEDS"] == len(SEEDS)).all():
        warnings.warn(
            "Not every participant has all five seeds. "
            "Inspect N_SEEDS before interpreting difficulty categories."
        )

    participant["ENSEMBLE_PRED_CLASS"] = (
        participant["FINAL_P_MEAN"] >= THRESHOLD
    ).astype(int)

    participant["ENSEMBLE_CORRECT"] = (
        participant["ENSEMBLE_PRED_CLASS"]
        == participant["TARGET"]
    ).astype(int)

    participant["SEED_ERROR_RATE"] = (
        1.0 - participant["SEED_CORRECT_RATE"]
    )

    participant["DISTANCE_FROM_0P5"] = (
        participant["FINAL_P_MEAN"] - THRESHOLD
    ).abs()

    participant["CORRECT_CLASS_PROBABILITY"] = np.where(
        participant["TARGET"] == 1,
        participant["FINAL_P_MEAN"],
        1.0 - participant["FINAL_P_MEAN"],
    )

    participant["DIFFICULTY_GROUP"] = np.select(
        [
            participant["N_SEEDS_CORRECT"] == 0,
            participant["N_SEEDS_CORRECT"] == participant["N_SEEDS"],
        ],
        [
            "consistently_wrong",
            "consistently_correct",
        ],
        default="seed_sensitive",
    )

    return participant

def load_oof_test_covariates(folds=FOLDS):
    """Collect only the held-out test participants from the prepared fold CSVs."""
    frames = []

    for fold in folds:
        path = (
            INPUT_ROOT
            / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
        )

        if not path.exists():
            raise FileNotFoundError(path)

        frame = pd.read_csv(path)

        require_columns(
            frame,
            ["RID", "DATA_ROLE", "MODEL_TARGET"],
            context=str(path),
        )

        test = frame.loc[
            frame["DATA_ROLE"].astype(str).str.lower() == "test"
        ].copy()

        test["FOLD_FROM_FILE"] = int(fold)

        if test["RID"].duplicated().any():
            raise ValueError(
                f"Duplicate test RID in prepared fold {fold}: {path}"
            )

        frames.append(test)

    oof = pd.concat(frames, ignore_index=True)

    if oof["RID"].duplicated().any():
        dup = oof.loc[oof["RID"].duplicated(keep=False), "RID"].tolist()
        raise ValueError(
            "The five held-out test folds are not disjoint. "
            f"Duplicate RIDs include: {dup[:20]}"
        )

    return oof

def standardized_mean_difference(x_a, x_b):
    """Absolute standardized mean difference for two numeric groups."""
    a = pd.to_numeric(x_a, errors="coerce").dropna().to_numpy(dtype=float)
    b = pd.to_numeric(x_b, errors="coerce").dropna().to_numpy(dtype=float)

    if len(a) < 2 or len(b) < 2:
        return np.nan

    var_a = np.var(a, ddof=1)
    var_b = np.var(b, ddof=1)
    pooled_sd = np.sqrt((var_a + var_b) / 2.0)

    if pooled_sd == 0 or not np.isfinite(pooled_sd):
        return np.nan

    return float((np.mean(a) - np.mean(b)) / pooled_sd)


## 2. Load the medium architecture and build one row per participant

The five seed predictions are averaged only for **diagnostic stability analysis**. This does not replace the original per-seed primary result.

The participant table records:

- seed-ensemble probability;
- seed-to-seed prediction SD;
- number of seeds correct;
- final / 3MT / TMC uncertainty where available;
- 3MT–TMC class disagreement where available;
- modality counts.


In [ ]:
medium_raw = load_neural_predictions(
    ARCHITECTURE_ROOTS[PRIMARY_ARCHITECTURE],
    PRIMARY_ARCHITECTURE,
)

print("Raw medium prediction rows:", len(medium_raw))
print("Unique participants:", medium_raw["RID"].nunique())
print("Seeds present:", sorted(medium_raw["SEED"].unique().tolist()))
print("Folds present:", sorted(medium_raw["FOLD"].unique().tolist()))
print("Prediction columns:")
print(medium_raw.columns.tolist())

medium_participant = aggregate_neural_by_participant(medium_raw)

print("\nParticipant-level table:")
display(medium_participant.head())

print("\nDifficulty group counts:")
display(
    medium_participant["DIFFICULTY_GROUP"]
    .value_counts()
    .rename_axis("DIFFICULTY_GROUP")
    .reset_index(name="N")
)

global_auc = safe_auc(
    medium_participant["TARGET"],
    medium_participant["FINAL_P_MEAN"],
)
global_ap = safe_ap(
    medium_participant["TARGET"],
    medium_participant["FINAL_P_MEAN"],
)

print(f"\nFive-seed probability-ensemble pooled ROC AUC: {global_auc:.6f}")
print(f"Five-seed probability-ensemble pooled AP:      {global_ap:.6f}")


## 3. Join the original held-out participant composition

This reconstructs the 544 out-of-fold test participants from the prepared fold tables and joins:

- branch-level modality availability;
- feature-level missingness masks;
- scaled continuous predictors;
- encoded categorical predictors;
- available audit labels.

No participant is removed.


In [ ]:
oof_covariates = load_oof_test_covariates()

print("OOF covariate participants:", len(oof_covariates))

# Verify fold and target alignment before joining.
alignment = medium_participant[
    ["RID", "TARGET", "FOLD"]
].merge(
    oof_covariates[
        ["RID", "MODEL_TARGET", "FOLD_FROM_FILE"]
    ],
    on="RID",
    how="outer",
    validate="one_to_one",
)

if alignment.isna().any().any():
    raise ValueError(
        "Participant sets do not align between predictions and prepared test folds."
    )

if not (
    alignment["TARGET"].astype(int)
    == alignment["MODEL_TARGET"].astype(int)
).all():
    raise ValueError("Prediction targets do not match prepared MODEL_TARGET.")

if not (
    alignment["FOLD"].astype(int)
    == alignment["FOLD_FROM_FILE"].astype(int)
).all():
    raise ValueError("Prediction fold labels do not match prepared test folds.")

branch_mask_cols = [
    c for c in oof_covariates.columns
    if c.startswith("BRANCH_MASK__")
]

feature_mask_cols = [
    c for c in oof_covariates.columns
    if c.startswith("FEATURE_MASK__")
]

scaled_continuous_cols = [
    c for c in oof_covariates.columns
    if c.endswith("__Z")
]

encoded_categorical_cols = [
    c for c in oof_covariates.columns
    if c.endswith("__IDX")
]

audit_cols = [
    c for c in [
        "PTID",
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ]
    if c in oof_covariates.columns
]

keep_covariates = (
    ["RID"]
    + audit_cols
    + branch_mask_cols
    + feature_mask_cols
    + scaled_continuous_cols
    + encoded_categorical_cols
)

keep_covariates = list(dict.fromkeys(keep_covariates))

participant = medium_participant.merge(
    oof_covariates[keep_covariates],
    on="RID",
    how="left",
    validate="one_to_one",
)

if feature_mask_cols:
    participant["OBSERVED_FEATURE_COUNT"] = (
        participant[feature_mask_cols].sum(axis=1)
    )
    participant["MISSING_FEATURE_COUNT"] = (
        len(feature_mask_cols)
        - participant["OBSERVED_FEATURE_COUNT"]
    )
    participant["FEATURE_AVAILABILITY_FRACTION"] = (
        participant["OBSERVED_FEATURE_COUNT"]
        / len(feature_mask_cols)
    )

if branch_mask_cols:
    participant["AVAILABLE_BRANCH_COUNT_FROM_INPUT"] = (
        participant[branch_mask_cols].sum(axis=1)
    )

print("Branch masks:", branch_mask_cols)
print("Number of feature masks:", len(feature_mask_cols))
print("Number of scaled continuous predictors:", len(scaled_continuous_cols))
print("Encoded categorical predictors:", encoded_categorical_cols)

display(participant.head())


## 4. Are the fold AUC differences larger than expected sampling uncertainty?

A fold contains only about 108–109 held-out participants. A visibly different point estimate does **not automatically mean** the fold contains a meaningfully different population.

This section therefore reports **stratified participant-bootstrap 95% CIs** for each fold's five-seed ensemble AUC. Overlapping wide intervals would indicate that at least part of the observed fold spread is compatible with finite-sample variation.


In [ ]:
fold_rows = []

for fold in FOLDS:
    g = participant.loc[participant["FOLD"] == fold].copy()

    boot = stratified_bootstrap_auc_ci(
        g["TARGET"],
        g["FINAL_P_MEAN"],
        n_bootstrap=N_BOOTSTRAP,
        random_seed=BOOTSTRAP_SEED + fold,
    )

    row = {
        "FOLD": fold,
        "N": len(g),
        "N_sMCI": int((g["TARGET"] == 0).sum()),
        "N_pMCI": int((g["TARGET"] == 1).sum()),
        "PMCI_RATE": float(g["TARGET"].mean()),
        "ENSEMBLE_ROC_AUC": boot["auc"],
        "AUC_CI_LOW": boot["ci_low"],
        "AUC_CI_HIGH": boot["ci_high"],
        "AUC_BOOTSTRAP_SD": boot["bootstrap_sd"],
        "ENSEMBLE_AP": safe_ap(g["TARGET"], g["FINAL_P_MEAN"]),
        "MEAN_SEED_PREDICTION_SD": float(g["FINAL_P_SD"].mean()),
        "MEAN_SEED_ERROR_RATE": float(g["SEED_ERROR_RATE"].mean()),
    }

    if "FINAL_UNCERTAINTY_MEAN" in g.columns:
        row["MEAN_FINAL_UNCERTAINTY"] = float(
            g["FINAL_UNCERTAINTY_MEAN"].mean()
        )

    if "MISSING_FEATURE_COUNT" in g.columns:
        row["MEAN_MISSING_FEATURE_COUNT"] = float(
            g["MISSING_FEATURE_COUNT"].mean()
        )

    for col in branch_mask_cols:
        row[f"AVAILABILITY__{col.replace('BRANCH_MASK__', '')}"] = float(
            g[col].mean()
        )

    fold_rows.append(row)

fold_difficulty = (
    pd.DataFrame(fold_rows)
    .sort_values("ENSEMBLE_ROC_AUC")
    .reset_index(drop=True)
)

display(fold_difficulty)

fold_difficulty.to_csv(
    TABLES_DIR / "fold_difficulty_bootstrap_and_composition.csv",
    index=False,
)

# AUC + bootstrap CI figure
plot_df = fold_difficulty.sort_values("FOLD")

fig, ax = plt.subplots(figsize=(8, 5))

y = plot_df["ENSEMBLE_ROC_AUC"].to_numpy()
lower = y - plot_df["AUC_CI_LOW"].to_numpy()
upper = plot_df["AUC_CI_HIGH"].to_numpy() - y

ax.errorbar(
    plot_df["FOLD"].astype(str),
    y,
    yerr=np.vstack([lower, upper]),
    fmt="o",
    capsize=4,
)

ax.set_xlabel("Outer test fold")
ax.set_ylabel("Five-seed ensemble ROC AUC")
ax.set_title("Fold ROC AUC with stratified participant-bootstrap 95% CI")
ax.set_ylim(0.70, 1.01)
fig.tight_layout()

figure_path = FIGURES_DIR / "fold_auc_bootstrap_ci.png"
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

print("Saved:", figure_path)


## 5. Which participants are difficult?

Definitions:

- **consistently correct** — all five seeds classify the participant correctly at 0.5;
- **consistently wrong** — all five seeds classify the participant incorrectly;
- **seed-sensitive** — some seeds are correct and others are wrong.

The `CORRECT_CLASS_PROBABILITY` is especially useful because it is target-aligned and threshold-independent as a *difficulty ranking*: lower values mean the ensemble placed less probability on the participant's true class.


In [ ]:
difficulty_summary = (
    participant
    .groupby("DIFFICULTY_GROUP", observed=True)
    .agg(
        N=("RID", "size"),
        PMCI_RATE=("TARGET", "mean"),
        MEAN_SEED_ERROR_RATE=("SEED_ERROR_RATE", "mean"),
        MEAN_PREDICTION_SD=("FINAL_P_SD", "mean"),
        MEAN_CORRECT_CLASS_PROBABILITY=("CORRECT_CLASS_PROBABILITY", "mean"),
    )
    .reset_index()
)

for col in [
    "FINAL_UNCERTAINTY_MEAN",
    "MISSING_FEATURE_COUNT",
    "AVAILABLE_BRANCH_COUNT_FROM_INPUT",
    "THREE_MT_TMC_CLASS_DISAGREE_RATE",
]:
    if col in participant.columns:
        extra = (
            participant
            .groupby("DIFFICULTY_GROUP", observed=True)[col]
            .mean()
            .rename(f"MEAN_{col}")
            .reset_index()
        )
        difficulty_summary = difficulty_summary.merge(
            extra,
            on="DIFFICULTY_GROUP",
            how="left",
        )

display(difficulty_summary)

difficulty_summary.to_csv(
    TABLES_DIR / "participant_difficulty_group_summary.csv",
    index=False,
)

# Hardest participants by target-aligned probability.
hardest_columns = [
    "RID",
    "FOLD",
    "TARGET",
    "DIFFICULTY_GROUP",
    "N_SEEDS_CORRECT",
    "FINAL_P_MEAN",
    "FINAL_P_SD",
    "CORRECT_CLASS_PROBABILITY",
    "SEED_ERROR_RATE",
]

for c in [
    "FINAL_UNCERTAINTY_MEAN",
    "TMC_UNCERTAINTY_MEAN",
    "THREE_MT_UNCERTAINTY_MEAN",
    "THREE_MT_TMC_CLASS_DISAGREE_RATE",
    "MISSING_FEATURE_COUNT",
    "AVAILABLE_BRANCH_COUNT_FROM_INPUT",
]:
    if c in participant.columns:
        hardest_columns.append(c)

hardest_participants = (
    participant[hardest_columns]
    .sort_values(
        ["CORRECT_CLASS_PROBABILITY", "FINAL_P_SD"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("Hardest 30 participants:")
display(hardest_participants.head(30))

hardest_participants.to_csv(
    TABLES_DIR / "participants_ranked_by_difficulty.csv",
    index=False,
)

# Difficulty categories by fold.
difficulty_by_fold = pd.crosstab(
    participant["FOLD"],
    participant["DIFFICULTY_GROUP"],
    normalize="index",
).reset_index()

display(difficulty_by_fold)

difficulty_by_fold.to_csv(
    TABLES_DIR / "difficulty_group_fraction_by_fold.csv",
    index=False,
)


## 6. Does missingness / modality availability explain difficulty?

This compares modality and feature availability across the three participant-difficulty groups. It is descriptive: no participant is excluded and no model is selected from these results.


In [ ]:
availability_columns = []

if "MISSING_FEATURE_COUNT" in participant.columns:
    availability_columns.extend([
        "MISSING_FEATURE_COUNT",
        "FEATURE_AVAILABILITY_FRACTION",
    ])

if "AVAILABLE_BRANCH_COUNT_FROM_INPUT" in participant.columns:
    availability_columns.append("AVAILABLE_BRANCH_COUNT_FROM_INPUT")

availability_columns.extend(branch_mask_cols)

if availability_columns:
    availability_summary = (
        participant
        .groupby("DIFFICULTY_GROUP", observed=True)[availability_columns]
        .mean()
        .T
    )
    display(availability_summary)

    availability_summary.to_csv(
        TABLES_DIR / "availability_by_difficulty_group.csv"
    )
else:
    print("No availability columns were found.")

# Direct consistently-wrong vs consistently-correct standardized differences.
wrong = participant.loc[
    participant["DIFFICULTY_GROUP"] == "consistently_wrong"
]
correct = participant.loc[
    participant["DIFFICULTY_GROUP"] == "consistently_correct"
]

availability_smd_rows = []

for col in availability_columns:
    smd = standardized_mean_difference(
        wrong[col],
        correct[col],
    )
    availability_smd_rows.append({
        "VARIABLE": col,
        "MEAN_CONSISTENTLY_WRONG": pd.to_numeric(
            wrong[col], errors="coerce"
        ).mean(),
        "MEAN_CONSISTENTLY_CORRECT": pd.to_numeric(
            correct[col], errors="coerce"
        ).mean(),
        "SMD_WRONG_MINUS_CORRECT": smd,
        "ABS_SMD": abs(smd) if np.isfinite(smd) else np.nan,
    })

availability_smd = (
    pd.DataFrame(availability_smd_rows)
    .sort_values("ABS_SMD", ascending=False)
    .reset_index(drop=True)
)

print("\nLargest availability differences:")
display(availability_smd.head(20))

availability_smd.to_csv(
    TABLES_DIR / "availability_smd_wrong_vs_correct.csv",
    index=False,
)


## 7. Which continuous participant characteristics differ most between consistently wrong and consistently correct cases?

The prepared continuous predictors are already fold-specific z-scored variables. This section reports **standardized mean differences (SMDs)** only as an exploratory ranking.

Large SMDs can suggest where difficult cases concentrate, but they are **not causal effects** and are not used here for feature selection.


In [ ]:
continuous_difference_rows = []

for col in scaled_continuous_cols:
    smd = standardized_mean_difference(
        wrong[col],
        correct[col],
    )

    continuous_difference_rows.append({
        "VARIABLE": col,
        "MEAN_CONSISTENTLY_WRONG": pd.to_numeric(
            wrong[col], errors="coerce"
        ).mean(),
        "MEAN_CONSISTENTLY_CORRECT": pd.to_numeric(
            correct[col], errors="coerce"
        ).mean(),
        "SMD_WRONG_MINUS_CORRECT": smd,
        "ABS_SMD": abs(smd) if np.isfinite(smd) else np.nan,
        "N_WRONG_NONMISSING": int(
            pd.to_numeric(wrong[col], errors="coerce").notna().sum()
        ),
        "N_CORRECT_NONMISSING": int(
            pd.to_numeric(correct[col], errors="coerce").notna().sum()
        ),
    })

continuous_differences = (
    pd.DataFrame(continuous_difference_rows)
    .sort_values("ABS_SMD", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 continuous-feature differences:")
display(continuous_differences.head(15))

continuous_differences.to_csv(
    TABLES_DIR / "continuous_feature_smd_wrong_vs_correct.csv",
    index=False,
)


## 8. Are the same participants difficult for small, medium, and full architectures?

If the same RIDs are repeatedly difficult across all three capacities, that supports the interpretation that the problem is **case / data difficulty rather than model size**.

If hard cases differ substantially by architecture, the errors are more architecture-specific.


In [ ]:
architecture_participants = {}

for architecture_name, root in ARCHITECTURE_ROOTS.items():
    try:
        raw = load_neural_predictions(
            root,
            architecture_name,
        )
        agg = aggregate_neural_by_participant(raw)
        architecture_participants[architecture_name] = agg
        print(
            f"Loaded {architecture_name}: "
            f"{len(agg)} participants, "
            f"ensemble AUC={safe_auc(agg['TARGET'], agg['FINAL_P_MEAN']):.6f}"
        )
    except FileNotFoundError as exc:
        print(f"Skipping {architecture_name}: {exc}")

if len(architecture_participants) >= 2:
    cross_arch = None

    for architecture_name, frame in architecture_participants.items():
        part = frame[
            [
                "RID",
                "TARGET",
                "FOLD",
                "ENSEMBLE_CORRECT",
                "SEED_CORRECT_RATE",
                "FINAL_P_MEAN",
                "FINAL_P_SD",
            ]
        ].copy()

        part = part.rename(
            columns={
                "ENSEMBLE_CORRECT":
                    f"{architecture_name}__ENSEMBLE_CORRECT",
                "SEED_CORRECT_RATE":
                    f"{architecture_name}__SEED_CORRECT_RATE",
                "FINAL_P_MEAN":
                    f"{architecture_name}__P_MEAN",
                "FINAL_P_SD":
                    f"{architecture_name}__P_SD",
            }
        )

        if cross_arch is None:
            cross_arch = part
        else:
            cross_arch = cross_arch.merge(
                part.drop(columns=["TARGET", "FOLD"]),
                on="RID",
                how="inner",
                validate="one_to_one",
            )

    correct_cols = [
        c for c in cross_arch.columns
        if c.endswith("__ENSEMBLE_CORRECT")
    ]

    cross_arch["N_ARCHITECTURES_WRONG"] = (
        len(correct_cols)
        - cross_arch[correct_cols].sum(axis=1)
    )

    print("\nHow many architectures get the participant wrong?")
    display(
        cross_arch["N_ARCHITECTURES_WRONG"]
        .value_counts()
        .sort_index()
        .rename_axis("N_ARCHITECTURES_WRONG")
        .reset_index(name="N_PARTICIPANTS")
    )

    cross_arch.to_csv(
        TABLES_DIR / "cross_architecture_participant_disagreement.csv",
        index=False,
    )

    # Pairwise overlap of ensemble errors.
    overlap_rows = []
    names = list(architecture_participants)

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a = names[i]
            b = names[j]

            a_wrong = set(
                architecture_participants[a]
                .loc[
                    architecture_participants[a]["ENSEMBLE_CORRECT"] == 0,
                    "RID",
                ]
                .tolist()
            )

            b_wrong = set(
                architecture_participants[b]
                .loc[
                    architecture_participants[b]["ENSEMBLE_CORRECT"] == 0,
                    "RID",
                ]
                .tolist()
            )

            intersection = len(a_wrong & b_wrong)
            union = len(a_wrong | b_wrong)

            overlap_rows.append({
                "ARCHITECTURE_A": a,
                "ARCHITECTURE_B": b,
                "A_WRONG": len(a_wrong),
                "B_WRONG": len(b_wrong),
                "BOTH_WRONG": intersection,
                "JACCARD_ERROR_OVERLAP":
                    intersection / union if union else np.nan,
            })

    architecture_error_overlap = pd.DataFrame(overlap_rows)
    display(architecture_error_overlap)

    architecture_error_overlap.to_csv(
        TABLES_DIR / "cross_architecture_error_overlap.csv",
        index=False,
    )
else:
    print("Fewer than two architecture result sets were available.")


## 9. Optional RBF-SVM disagreement / rescue analysis

The earlier classical baseline notebook saved files named:

`svm_rbf_fold_<fold>_seed_<seed>_test_predictions.csv`

with columns `RID`, `TARGET`, and `PROB_PMCI`.

This cell searches a few likely JHub locations. If it cannot find the files, set `SVM_PREDICTIONS_DIR` manually to the directory containing those CSVs and rerun the cell.

This analysis asks a particularly useful question:

> **When the medium 3MT–TMC hybrid is wrong, is the SVM right?**

If yes, the models make complementary errors and an SVM–evidential fusion experiment has a stronger empirical motivation.


In [ ]:
# Optional explicit SVM output path.
SVM_PREDICTIONS_DIR = None

SVM_SEEDS = [17, 42, 73]
SVM_RUN_TAG = "standard_ml_baselines_mri_embeddings_reasonable_v1"

candidate_svm_dirs = [
    (
        PROJECT_ROOT
        / "models"
        / "3mt_tmc_evidential"
        / "standard_ml_baselines"
        / SVM_RUN_TAG
        / "fold_predictions"
    ),
    (
        RUNS_ROOT
        / "standard_ml_baselines"
        / SVM_RUN_TAG
        / "fold_predictions"
    ),
    (
        RUNS_ROOT
        / SVM_RUN_TAG
        / "fold_predictions"
    ),
]

if SVM_PREDICTIONS_DIR is None:
    for candidate in candidate_svm_dirs:
        probe = candidate / "svm_rbf_fold_0_seed_17_test_predictions.csv"
        if probe.exists():
            SVM_PREDICTIONS_DIR = candidate
            break

if SVM_PREDICTIONS_DIR is None:
    print("SVM prediction files were not auto-discovered.")
    print(
        "Set SVM_PREDICTIONS_DIR to the folder containing "
        "svm_rbf_fold_0_seed_17_test_predictions.csv and rerun this cell."
    )
else:
    SVM_PREDICTIONS_DIR = Path(SVM_PREDICTIONS_DIR)
    print("Using SVM predictions:", SVM_PREDICTIONS_DIR)

    svm_frames = []

    for seed in SVM_SEEDS:
        for fold in FOLDS:
            path = (
                SVM_PREDICTIONS_DIR
                / f"svm_rbf_fold_{fold}_seed_{seed}_test_predictions.csv"
            )

            if not path.exists():
                raise FileNotFoundError(path)

            frame = pd.read_csv(path)
            require_columns(
                frame,
                ["RID", "TARGET", "PROB_PMCI"],
                str(path),
            )

            frame = frame.copy()
            frame["SEED"] = seed
            frame["FOLD"] = fold
            svm_frames.append(frame)

    svm_raw = pd.concat(svm_frames, ignore_index=True)

    if svm_raw.duplicated(["SEED", "RID"]).any():
        raise ValueError("Duplicate RID within an SVM seed's OOF predictions.")

    svm_raw["PRED_CLASS"] = (
        svm_raw["PROB_PMCI"] >= THRESHOLD
    ).astype(int)
    svm_raw["CORRECT"] = (
        svm_raw["PRED_CLASS"] == svm_raw["TARGET"]
    ).astype(int)

    svm_participant = (
        svm_raw
        .groupby("RID")
        .agg(
            TARGET=("TARGET", "first"),
            FOLD=("FOLD", "first"),
            N_SVM_SEEDS=("SEED", "nunique"),
            SVM_P_MEAN=("PROB_PMCI", "mean"),
            SVM_P_SD=("PROB_PMCI", "std"),
            SVM_SEED_CORRECT_RATE=("CORRECT", "mean"),
        )
        .reset_index()
    )

    svm_participant["SVM_ENSEMBLE_PRED_CLASS"] = (
        svm_participant["SVM_P_MEAN"] >= THRESHOLD
    ).astype(int)

    svm_participant["SVM_ENSEMBLE_CORRECT"] = (
        svm_participant["SVM_ENSEMBLE_PRED_CLASS"]
        == svm_participant["TARGET"]
    ).astype(int)

    print(
        "SVM three-seed probability-ensemble pooled AUC:",
        f"{safe_auc(svm_participant['TARGET'], svm_participant['SVM_P_MEAN']):.6f}",
    )

    neural_svm = participant.merge(
        svm_participant,
        on="RID",
        how="inner",
        suffixes=("_NEURAL", "_SVM"),
        validate="one_to_one",
    )

    if not (
        neural_svm["TARGET_NEURAL"].astype(int)
        == neural_svm["TARGET_SVM"].astype(int)
    ).all():
        raise ValueError("Neural and SVM targets do not match.")

    neural_svm["MODEL_DISAGREEMENT_GROUP"] = np.select(
        [
            (neural_svm["ENSEMBLE_CORRECT"] == 1)
            & (neural_svm["SVM_ENSEMBLE_CORRECT"] == 1),

            (neural_svm["ENSEMBLE_CORRECT"] == 0)
            & (neural_svm["SVM_ENSEMBLE_CORRECT"] == 1),

            (neural_svm["ENSEMBLE_CORRECT"] == 1)
            & (neural_svm["SVM_ENSEMBLE_CORRECT"] == 0),
        ],
        [
            "both_correct",
            "svm_rescues_neural",
            "neural_rescues_svm",
        ],
        default="both_wrong",
    )

    disagreement_counts = (
        neural_svm["MODEL_DISAGREEMENT_GROUP"]
        .value_counts()
        .rename_axis("GROUP")
        .reset_index(name="N")
    )

    display(disagreement_counts)

    diagnostic_cols = [
        c for c in [
            "FINAL_UNCERTAINTY_MEAN",
            "FINAL_P_SD",
            "MISSING_FEATURE_COUNT",
            "AVAILABLE_BRANCH_COUNT_FROM_INPUT",
            "THREE_MT_TMC_CLASS_DISAGREE_RATE",
        ]
        if c in neural_svm.columns
    ]

    if diagnostic_cols:
        disagreement_profile = (
            neural_svm
            .groupby("MODEL_DISAGREEMENT_GROUP", observed=True)[diagnostic_cols]
            .mean()
        )
        display(disagreement_profile)

        disagreement_profile.to_csv(
            TABLES_DIR / "svm_neural_disagreement_profile.csv"
        )

    neural_svm.to_csv(
        TABLES_DIR / "svm_neural_participant_disagreement.csv",
        index=False,
    )

    disagreement_counts.to_csv(
        TABLES_DIR / "svm_neural_disagreement_counts.csv",
        index=False,
    )


## 10. Final diagnostic summary

This cell prints a compact set of quantities to send back for interpretation. It deliberately avoids a rule such as “remove these participants.” The scientific question is whether difficult cases share identifiable characteristics and whether those characteristics are consistent across architectures / SVM.


In [ ]:
print("=" * 80)
print("PARTICIPANT / FOLD DIFFICULTY DIAGNOSTIC SUMMARY")
print("=" * 80)

print(f"Participants analysed: {len(participant)}")
print(
    "Medium five-seed ensemble pooled ROC AUC:",
    f"{safe_auc(participant['TARGET'], participant['FINAL_P_MEAN']):.6f}",
)

print("\nDifficulty groups:")
print(
    participant["DIFFICULTY_GROUP"]
    .value_counts()
    .to_string()
)

print("\nFold AUCs with bootstrap 95% CIs:")
print(
    fold_difficulty[
        [
            "FOLD",
            "N",
            "PMCI_RATE",
            "ENSEMBLE_ROC_AUC",
            "AUC_CI_LOW",
            "AUC_CI_HIGH",
            "MEAN_SEED_ERROR_RATE",
        ]
    ].to_string(index=False)
)

if "MISSING_FEATURE_COUNT" in participant.columns:
    print("\nMean missing feature count by difficulty group:")
    print(
        participant
        .groupby("DIFFICULTY_GROUP")["MISSING_FEATURE_COUNT"]
        .mean()
        .to_string()
    )

if "FINAL_UNCERTAINTY_MEAN" in participant.columns:
    print("\nMean final uncertainty by difficulty group:")
    print(
        participant
        .groupby("DIFFICULTY_GROUP")["FINAL_UNCERTAINTY_MEAN"]
        .mean()
        .to_string()
    )

print("\nTop continuous variables separating consistently wrong vs correct:")
if len(continuous_differences):
    print(
        continuous_differences[
            ["VARIABLE", "SMD_WRONG_MINUS_CORRECT", "ABS_SMD"]
        ]
        .head(10)
        .to_string(index=False)
    )

print("\nSaved tables:")
for path in sorted(TABLES_DIR.glob("*.csv")):
    print(" -", path)

print("\nSaved figures:")
for path in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", path)

print(
    "\nInterpretation reminder: this is post-hoc outer-test analysis. "
    "Use it to understand failure modes and motivate future/external validation, "
    "not to delete difficult participants or re-select a model on the same test data."
)
